In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'triplet.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

# Calculating AVG

In [3]:
expansions = {
    'estimated_p': ['p0', 'p1'],
    's_': ['s0', 's1', 's2', 's3'],
    's__': ['s_0', 's_1', 's_2', 's_3'],
}

df = get_data_expanded(data.evaluation_data, expansions)
df_avgs = df.groupby('episode')[['p0', 'p1']].mean().reset_index()

def get_avg_per_epi(row):
    return df_avgs.loc[df_avgs['episode'] == row.episode][['p0', 'p1']].values[0]
df[['avg_p0', 'avg_p1']] = df.apply(get_avg_per_epi, axis=1, result_type='expand')

df[['episode', 'avg_p0', 'avg_p1']]

,episode,avg_p0,avg_p1
0,0,1.466500,-2.369667
1,0,1.466500,-2.369667
2,0,1.466500,-2.369667
3,0,1.466500,-2.369667
4,0,1.466500,-2.369667
...,...,...,...
2122,99,0.219067,0.880600
2123,99,0.219067,0.880600
2124,99,0.219067,0.880600
2125,99,0.219067,0.880600


In [4]:
df_std = df.groupby('episode')[['p0', 'p1']].std().reset_index()
df_min = df.groupby('episode')[['p0', 'p1']].min().reset_index()
df_max = df.groupby('episode')[['p0', 'p1']].max().reset_index()
df_count = df.groupby('episode')[['p0', 'p1']].count().reset_index()
df_stats = df_avgs[['episode']].copy()

df_stats[['avg_p0', 'avg_p1']] = df_avgs[['p0', 'p1']]
df_stats[['std_p0', 'std_p1']] = df_std[['p0', 'p1']]
df_stats[['min_p0', 'min_p1']] = df_min[['p0', 'p1']]
df_stats[['max_p0', 'max_p1']] = df_max[['p0', 'p1']]
df_stats['range_p0'] = df_stats['max_p0'] - df_stats['min_p0']
df_stats['range_p1'] = df_stats['max_p1'] - df_stats['min_p1']
df_stats[['count']] = df_count[['p0']]

df_stats

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
0,0,1.466500,-2.369667,0.169200,0.445541,1.219,-3.203,1.687,-1.872,0.468,1.331,6
1,1,0.326556,0.677556,0.019391,0.047342,0.279,0.608,0.343,0.765,0.064,0.157,9
2,2,0.256688,0.729563,0.058291,0.074789,0.164,0.603,0.376,0.862,0.212,0.259,16
3,3,0.176211,0.981632,0.040327,0.036797,0.112,0.932,0.237,1.053,0.125,0.121,19
4,4,0.349615,0.395846,0.028707,0.095779,0.313,0.204,0.403,0.501,0.090,0.297,13
...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,1.017067,-1.468200,0.076372,0.183177,0.869,-1.740,1.167,-1.172,0.298,0.568,15
96,96,0.195895,0.971500,0.028781,0.050625,0.122,0.858,0.236,1.069,0.114,0.211,38
97,97,0.353371,0.494171,0.050469,0.111645,0.274,0.355,0.523,0.903,0.249,0.548,35
98,98,0.283277,0.757289,0.054413,0.105120,0.173,0.591,0.428,1.018,0.255,0.427,83


In [5]:
df_stats.describe()

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,49.500000,0.511578,0.045363,0.052700,0.139583,0.428350,-0.172990,0.602460,0.264420,0.174110,0.437410,21.270000
std,29.011492,0.473653,1.338915,0.039009,0.151087,0.443566,1.531132,0.513986,1.178785,0.106632,0.399525,14.489882
min,0.000000,0.074631,-5.706429,0.013852,0.028886,0.020000,-6.970000,0.137000,-5.026000,0.042000,0.101000,3.000000
25%,24.750000,0.227411,-0.345859,0.026392,0.052781,0.161250,-0.579000,0.287750,-0.067500,0.093750,0.197500,11.000000
50%,49.500000,0.346462,0.584163,0.041872,0.080400,0.276000,0.461000,0.410500,0.726000,0.143500,0.284000,16.500000
75%,74.250000,0.607573,0.880541,0.065802,0.172787,0.522750,0.752000,0.703750,0.961000,0.237750,0.553000,29.250000
max,99.000000,2.724571,1.351846,0.248000,1.006422,2.596000,1.247000,2.856000,1.519000,0.496000,2.662000,83.000000


# Infering with avg

In [6]:
import torch
import torch.nn as nn

m = model.transition_estimator.state_layer

input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'avg_p0', 'avg_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0315)

In [7]:
results = data.get_evaluation_metrics()
results.rse.mean()

np.float64(0.20761871180065822)

# Optimazing p

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

history = []

for epi, p1, p2 in df_avgs.values:
    print(f'{epi:}')
    d = df[df['episode'] == epi].copy().reset_index(drop=True)

    target_value = torch.tensor(d[['s_0', 's_1', 's_2', 's_3']].values)
    param = torch.tensor([p1, p2], requires_grad=True)
    print(f'initial value for input Param: {[round(p,4) for p in param.tolist()]}') 
    input_values = torch.tensor(d[['s0', 's1', 's2', 's3', 'a_']].values) 

    learning_rate = 0.1
    num_epochs = 500

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.repeat((input_values.shape[0], 1))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        # loss = criterion(output.float(), target_value.float())
        
        def normilize(v): 
            mins = input_values[:,:-1].min(axis=0).values.repeat((v.shape[0], 1))
            maxs = input_values[:,:-1].max(axis=0).values.repeat((v.shape[0], 1))
            rang = maxs - mins
            return (v - mins) / rang
        open_loss = criterion(normilize(output).float(), normilize(target_value).float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        # loss = torch.sqrt(open_loss.mean(axis=1).sum())
        # loss = open_loss.mean()
        # loss = open_loss.sum()

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # history.append((epoch, param.tolist(), loss.item()))

        # if (epoch + 1) % 100 == 0:
        #     # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
        #     print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
    history.append({'episode': epi, 'optimized_p': param.tolist(), 'loss':  loss.item()})
    print(f'Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')


0.0
initial value for input Param: [1.4665, -2.3697]
Loss: 0.5950, Input Param: [0.7919, -0.7299]
1.0
initial value for input Param: [0.3266, 0.6776]
Loss: 0.0360, Input Param: [0.7028, -0.7253]
2.0
initial value for input Param: [0.2567, 0.7296]
Loss: 0.0749, Input Param: [0.1111, 1.3475]
3.0
initial value for input Param: [0.1762, 0.9816]
Loss: 0.1125, Input Param: [0.1067, 1.3394]
4.0
initial value for input Param: [0.3496, 0.3958]
Loss: 0.0420, Input Param: [-0.0089, 1.7045]
5.0
initial value for input Param: [0.1868, 1.05]
Loss: 0.0932, Input Param: [-0.0554, 1.6579]
6.0
initial value for input Param: [0.1568, 1.129]
Loss: 0.2336, Input Param: [-0.428, 2.9977]
7.0
initial value for input Param: [0.5134, -0.075]
Loss: 0.0999, Input Param: [0.0381, 1.6504]
8.0
initial value for input Param: [1.2798, -2.146]
Loss: 0.2510, Input Param: [0.6245, -0.2971]
9.0
initial value for input Param: [0.282, 0.6645]
Loss: 0.0339, Input Param: [-0.0266, 1.6719]
10.0
initial value for input Param: [

In [9]:
import pandas as pd
df_optim = pd.DataFrame(history)


expansions = {
    'optimized_p': ['opt_p0', 'opt_p1'],
}

df_optim = get_data_expanded(df_optim, expansions)[['episode', 'loss', 'opt_p0', 'opt_p1']]
df_optim.loss.mean()

np.float64(0.172600389290601)

In [10]:
import torch
import torch.nn as nn

def get_opt_per_epi(row):
    return df_optim.loc[df_optim['episode'] == row.episode][['opt_p0', 'opt_p1']].values[0]
df[['opt_p0', 'opt_p1']] = df.apply(get_opt_per_epi, axis=1, result_type='expand')


input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'opt_p0', 'opt_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0293)

In [11]:
del model
del data